In [2]:
from google.colab import files
uploaded = files.upload()

import os, zipfile

os.makedirs("/root/.kaggle", exist_ok=True)
for fn in uploaded.keys():
    os.rename(fn, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Step 2: Download the competition data
!kaggle competitions download -c playground-series-s6e1

#!/bin/bash
!kaggle datasets download kundanbedmutha/exam-score-prediction-dataset

# Step 4: Unzip them
# Unzip competition
with zipfile.ZipFile("playground-series-s6e1.zip", "r") as z:
    z.extractall("playground-series-s6e1")

# # Unzip the simulated roads accident dataset (it will produce those CSVs)
# with zipfile.ZipFile("simulated-roads-accident-data.zip", "r") as z:
#     z.extractall("simulated-roads-accident-data")

print("ps6e1 + examdataset")

Saving kaggle.json to kaggle.json
  0% 0.00/13.8M [00:00<?, ?B/s]
100% 13.8M/13.8M [00:00<00:00, 1.27GB/s]
Dataset URL: https://www.kaggle.com/datasets/kundanbedmutha/exam-score-prediction-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
  0% 0.00/318k [00:00<?, ?B/s]
100% 318k/318k [00:00<00:00, 636MB/s]
ps6e1 + examdataset


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
train_data = pd.read_csv("playground-series-s6e1/train.csv")
test_data = pd.read_csv("playground-series-s6e1/test.csv")

In [6]:
train_data.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [27]:
df = train_data.drop(columns=['id'], axis=1)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   age               630000 non-null  int64  
 1   gender            630000 non-null  object 
 2   course            630000 non-null  object 
 3   study_hours       630000 non-null  float64
 4   class_attendance  630000 non-null  float64
 5   internet_access   630000 non-null  object 
 6   sleep_hours       630000 non-null  float64
 7   sleep_quality     630000 non-null  object 
 8   study_method      630000 non-null  object 
 9   facility_rating   630000 non-null  object 
 10  exam_difficulty   630000 non-null  object 
 11  exam_score        630000 non-null  float64
dtypes: float64(4), int64(1), object(7)
memory usage: 57.7+ MB


In [29]:
df.describe()

,age,study_hours,class_attendance,sleep_hours,exam_score
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,20.545821,4.002337,71.987261,7.072758,62.506672
std,2.260238,2.359880,17.430098,1.744811,18.916884
min,17.000000,0.080000,40.600000,4.100000,19.599000
25%,19.000000,1.970000,57.000000,5.600000,48.800000
50%,21.000000,4.000000,72.600000,7.100000,62.600000
75%,23.000000,6.050000,87.200000,8.600000,76.300000
max,24.000000,7.910000,99.400000,9.900000,100.000000


In [30]:
for objCol in df.select_dtypes(include='object').columns:
  print(objCol, ' - ', df[objCol].unique())

gender  -  ['female' 'other' 'male']
course  -  ['b.sc' 'diploma' 'bca' 'b.com' 'ba' 'bba' 'b.tech']
internet_access  -  ['no' 'yes']
sleep_quality  -  ['average' 'poor' 'good']
study_method  -  ['online videos' 'self-study' 'coaching' 'group study' 'mixed']
facility_rating  -  ['low' 'medium' 'high']
exam_difficulty  -  ['easy' 'moderate' 'hard']


In [42]:
df1 = df.copy()

In [43]:
labelEncoderMap = {};

In [44]:
from sklearn.preprocessing import LabelEncoder

In [45]:
for objCol in df1.select_dtypes(include='object').columns:
  le = LabelEncoder()
  df1[objCol] = le.fit_transform(df1[objCol])
  labelEncoderMap[objCol] = le
  print(objCol, ' - ', df1[objCol].unique())

gender  -  [0 2 1]
course  -  [1 6 5 0 3 4 2]
internet_access  -  [0 1]
sleep_quality  -  [0 2 1]
study_method  -  [3 4 0 1 2]
facility_rating  -  [1 2 0]
exam_difficulty  -  [0 2 1]


In [46]:
df1.head()

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,21,0,1,7.91,98.8,0,4.9,0,3,1,0,78.3
1,18,2,6,4.95,94.8,1,4.7,2,4,2,2,46.7
2,20,0,1,4.68,92.6,1,5.8,2,0,0,2,99.0
3,19,1,1,2.00,49.5,1,8.3,0,1,0,2,63.9
4,23,1,5,7.65,86.9,1,9.6,1,4,0,0,100.0


In [51]:
for col, le in labelEncoderMap.items():
    assert set(df[col]).issubset(set(le.classes_)), f"Unseen labels in {col}"

In [52]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [58]:
X = df1.drop(columns=['exam_score'], axis=1)
y = df1['exam_score']

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

In [60]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((535500, 11), (94500, 11), (535500,), (94500,))

In [61]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [62]:
lr.score(X_test, y_test)

0.7230043949983388

In [63]:
root_mean_squared_error(y_test, lr.predict(X_test))

9.917732941363834

In [74]:
import xgboost as xgb

In [77]:
xgbModel = xgb.XGBRegressor()
xgbModel.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [78]:
xgbModel.score(X_test, y_test), root_mean_squared_error(y_test, xgbModel.predict(X_test))

(0.782825125945702, 8.781743737875974)

In [ ]:
x

In [64]:
print("---------------------")

---------------------


In [65]:
test_data.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate


In [66]:
test_data.select_dtypes(include='object').columns == df.select_dtypes(include='object').columns

array([ True,  True,  True,  True,  True,  True,  True])

In [67]:
for objCol in test_data.select_dtypes(include='object').columns:
  test_data[objCol] = labelEncoderMap[objCol].transform(test_data[objCol])

In [68]:
test_data.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,2,3,6.85,65.2,1,5.2,2,1,0,0
1,630001,18,1,6,6.61,45.0,0,9.3,2,0,1,0
2,630002,24,0,2,6.60,98.5,1,6.2,1,1,2,2
3,630003,24,1,6,3.03,66.3,1,5.7,0,2,2,2
4,630004,20,0,2,2.03,42.4,1,9.2,0,0,1,2


In [70]:
final_test_data = test_data.drop(columns=['id'], axis=1)
id = test_data['id']

In [79]:
# lr.predict(final_test_data)
xgbModel.predict(final_test_data)

array([72.24985 , 71.147354, 88.19859 , ..., 90.0829  , 54.100903,
       67.80677 ], dtype=float32)

In [80]:
submission = pd.DataFrame({'id': id, 'exam_score': xgbModel.predict(final_test_data)})

In [81]:
submission.to_csv('submission.csv', index=False)